# UPR Bootstrap — Write Missing `upr/` Files to Drive

**Run this notebook ONCE before running Notebooks 01–04.**

This cell writes all updated/new `upr/` Python modules directly to your Google Drive project directory.  
No git pull, no pip install required.

In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

UPR_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime/upr'
os.makedirs(UPR_DIR, exist_ok=True)

files = {}

# ── upr/__init__.py ──────────────────────────────────────────────────────────
files['__init__.py'] = '''\
from .bit_ops import (
    float16_to_uint16_numpy,
    uint16_to_float16_torch,
    extract_bit_plane_np,
    pack_bit_plane,
    unpack_bit_plane,
    reconstruct_tensor,
    get_packing_stats,
)
from .converter import convert_to_bitplanes
from .loader import BitPlaneModel
from .numerical import (
    compute_cosine_similarity,
    compute_kl_divergence,
    compute_numerical_metrics,
)
from .metrics import compute_weight_metrics
from .metadata import set_seed, collect_experiment_metadata, get_git_commit_hash
from .profiler import IsolatedTimer, MemoryProfiler
from .layer_hooks import LayerActivationCollector, compare_layer_activations

__version__ = "0.1.0"
__all__ = [
    "float16_to_uint16_numpy",
    "uint16_to_float16_torch",
    "extract_bit_plane_np",
    "pack_bit_plane",
    "unpack_bit_plane",
    "reconstruct_tensor",
    "get_packing_stats",
    "convert_to_bitplanes",
    "BitPlaneModel",
    "compute_weight_metrics",
    "compute_cosine_similarity",
    "compute_kl_divergence",
    "compute_numerical_metrics",
    "set_seed",
    "collect_experiment_metadata",
    "get_git_commit_hash",
    "IsolatedTimer",
    "MemoryProfiler",
    "LayerActivationCollector",
    "compare_layer_activations",
]
'''

# ── upr/bit_ops.py ───────────────────────────────────────────────────────────
files['bit_ops.py'] = '''\
import torch
import numpy as np
import math
import gc
from typing import Tuple, Dict, Optional, Union, Any

def float16_to_uint16_numpy(tensor: torch.Tensor) -> np.ndarray:
    """Reinterprets float16 torch.Tensor as uint16 numpy ndarray (bit-exact)."""
    np_f16 = tensor.detach().cpu().to(torch.float16).numpy()
    return np_f16.view(np.uint16)

def uint16_to_float16_torch(np_uint16: np.ndarray, device: Union[str, torch.device] = \'cpu\') -> torch.Tensor:
    """Reinterprets uint16 numpy ndarray as float16 torch.Tensor (bit-exact)."""
    np_f16 = np_uint16.view(np.float16)
    tensor = torch.from_numpy(np_f16).to(device)
    if torch.isnan(tensor).any() or torch.isinf(tensor).any():
        tensor = torch.nan_to_num(tensor, nan=0.0, posinf=65504.0, neginf=-65504.0)
    return tensor

def extract_bit_plane_np(uint16_arr: np.ndarray, bit_index: int) -> np.ndarray:
    assert 0 <= bit_index <= 15, f"bit_index must be between 0 and 15, got {bit_index}"
    return ((uint16_arr >> bit_index) & 1).astype(np.uint8)

def pack_bit_plane(bit_arr: np.ndarray) -> bytes:
    flat = bit_arr.ravel()
    num_elements = flat.size
    packed = np.packbits(flat, bitorder=\'big\')
    packed_bytes = packed.tobytes()
    expected_bytes = math.ceil(num_elements / 8)
    assert len(packed_bytes) == expected_bytes, (
        f"Bit packing assertion failed: expected {expected_bytes} bytes for {num_elements} bits, got {len(packed_bytes)}"
    )
    return packed_bytes

def unpack_bit_plane(packed_bytes: bytes, num_elements: int, shape: Optional[Tuple[int, ...]] = None) -> np.ndarray:
    packed_np = np.frombuffer(packed_bytes, dtype=np.uint8)
    unpacked = np.unpackbits(packed_np, bitorder=\'big\')[:num_elements]
    if shape is not None:
        unpacked = unpacked.reshape(shape)
    return unpacked.astype(np.uint8)

def get_packing_stats(num_elements: int, bits_reconstructed: int) -> Dict[str, Any]:
    raw_fp16_bytes = num_elements * 2
    packed_bits_bytes = math.ceil(num_elements / 8) * bits_reconstructed
    compression_ratio = raw_fp16_bytes / packed_bits_bytes if packed_bits_bytes > 0 else 0.0
    return {
        "num_elements": num_elements,
        "bits_reconstructed": bits_reconstructed,
        "raw_fp16_bytes": raw_fp16_bytes,
        "packed_bits_bytes": packed_bits_bytes,
        "compression_ratio": round(compression_ratio, 4)
    }

def reconstruct_tensor(
    planes_dict: Dict[int, bytes],
    selected_bits: int,
    original_shape: Tuple[int, ...],
    device: Union[str, torch.device] = \'cpu\'
) -> torch.Tensor:
    assert 1 <= selected_bits <= 16, f"selected_bits must be between 1 and 16, got {selected_bits}"
    num_elements = int(np.prod(original_shape)) if len(original_shape) > 0 else 1
    accum = np.zeros(num_elements, dtype=np.uint32)
    start_bit = 15
    end_bit = 16 - selected_bits
    for b in range(start_bit, end_bit - 1, -1):
        if b in planes_dict:
            bits = unpack_bit_plane(planes_dict[b], num_elements)
            accum |= (bits.astype(np.uint32) << b)
            del bits
    uint16_arr = accum.astype(np.uint16).reshape(original_shape)
    del accum
    tensor = uint16_to_float16_torch(uint16_arr, device=device)
    del uint16_arr
    return tensor
'''

# ── upr/numerical.py ─────────────────────────────────────────────────────────
files['numerical.py'] = '''\
import torch
import torch.nn.functional as F
import numpy as np
from typing import Dict, Any

def compute_cosine_similarity(t1: torch.Tensor, t2: torch.Tensor) -> float:
    """Fix 1 — Verified cosine similarity using F.cosine_similarity on float32.
    Enforces mathematical bounds: -1.0 - 1e-6 <= cosine <= 1.0 + 1e-6."""
    flat1 = t1.detach().to(device="cpu", dtype=torch.float32).reshape(1, -1)
    flat2 = t2.detach().to(device="cpu", dtype=torch.float32).reshape(1, -1)
    norm1 = torch.norm(flat1)
    norm2 = torch.norm(flat2)
    if norm1 == 0 and norm2 == 0:
        return 1.0
    elif norm1 == 0 or norm2 == 0:
        return 0.0
    sim = float(F.cosine_similarity(flat1, flat2, dim=1).item())
    assert sim <= 1.0 + 1e-6, f"Cosine similarity exceeds upper bound: {sim}"
    assert sim >= -1.0 - 1e-6, f"Cosine similarity below lower bound: {sim}"
    return max(-1.0, min(1.0, sim))

def compute_kl_divergence(p_logits: torch.Tensor, q_logits: torch.Tensor) -> float:
    p_f32 = p_logits.detach().to(device="cpu", dtype=torch.float32)
    q_f32 = q_logits.detach().to(device="cpu", dtype=torch.float32)
    p_log_prob = F.log_softmax(p_f32, dim=-1)
    q_log_prob = F.log_softmax(q_f32, dim=-1)
    kl = F.kl_div(q_log_prob, p_log_prob, log_target=True, reduction="batchmean")
    val = float(kl.item())
    return val if not (np.isnan(val) or np.isinf(val)) else 0.0

def compute_numerical_metrics(original: torch.Tensor, reconstructed: torch.Tensor) -> Dict[str, Any]:
    """Fix 2 — Numerical Validation Framework: MAE, RMSE, Max Abs Error, MRE, CosSim, KL."""
    orig_f32 = original.detach().to(device="cpu", dtype=torch.float32)
    recon_f32 = reconstructed.detach().to(device="cpu", dtype=torch.float32)
    is_exact = bool(torch.equal(original.detach().cpu(), reconstructed.detach().cpu()))
    diff = torch.abs(orig_f32 - recon_f32)
    mae = float(diff.mean().item())
    rmse = float(torch.sqrt(torch.mean((orig_f32 - recon_f32) ** 2)).item())
    max_abs_error = float(diff.max().item())
    denom = torch.abs(orig_f32) + 1e-8
    mre = float((diff / denom).mean().item())
    cos_sim = compute_cosine_similarity(orig_f32, recon_f32)
    kl_div = 0.0
    if orig_f32.ndim >= 2 and orig_f32.shape[-1] > 1:
        kl_div = compute_kl_divergence(orig_f32, recon_f32)
    return {
        "torch_equal": is_exact,
        "mae": mae,
        "rmse": rmse,
        "max_abs_error": max_abs_error,
        "mean_relative_error": mre,
        "cosine_similarity": cos_sim,
        "kl_divergence": kl_div,
        "num_elements": int(orig_f32.numel())
    }
'''

# ── upr/metadata.py ──────────────────────────────────────────────────────────
files['metadata.py'] = '''\
import os
import sys
import random
import subprocess
import time
import torch
import numpy as np
from typing import Dict, Any, Optional

def set_seed(seed: int = 42) -> None:
    """Fix 10 — Fixes Python, NumPy, PyTorch CPU/CUDA, cuDNN random seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_git_commit_hash() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "unknown"

def collect_experiment_metadata(
    precision_bits: int,
    dataset_name: str = "wikitext-2-raw-v1",
    seed: int = 42,
    model_name: str = "Qwen/Qwen3.5-0.8B",
    extra_info: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:
    """Fix 11 — Stores Git commit, timestamp, representation, precision, dataset, seed, GPU info."""
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    meta = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "git_commit": get_git_commit_hash(),
        "model_name": model_name,
        "representation": "bitplane_packed_uint16",
        "precision_bits": precision_bits,
        "dataset": dataset_name,
        "seed": seed,
        "python_version": sys.version.split()[0],
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu_device": gpu_info,
    }
    if extra_info:
        meta.update(extra_info)
    return meta
'''

# ── upr/profiler.py ──────────────────────────────────────────────────────────
files['profiler.py'] = '''\
import os
import time
import csv
import psutil
import torch
from typing import Dict, Any

class IsolatedTimer:
    """Fix 6 — Measures isolated phase durations (never combined)."""
    def __init__(self):
        self.timers: Dict[str, float] = {}
        self._starts: Dict[str, float] = {}

    def start(self, phase_name: str) -> None:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self._starts[phase_name] = time.perf_counter()

    def stop(self, phase_name: str) -> float:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = self._starts.pop(phase_name, time.perf_counter())
        elapsed = time.perf_counter() - start
        self.timers[phase_name] = elapsed
        return elapsed

    def get_summary(self) -> Dict[str, float]:
        return dict(self.timers)


class MemoryProfiler:
    """Fix 7 — Records CPU RAM, GPU VRAM, checkpoint sizes to results/memory.csv."""
    def __init__(self):
        self.process = psutil.Process(os.getpid())

    def get_cpu_ram_mb(self) -> float:
        return self.process.memory_info().rss / (1024 * 1024)

    def get_gpu_vram_mb(self) -> float:
        if torch.cuda.is_available():
            return torch.cuda.memory_allocated() / (1024 * 1024)
        return 0.0

    def get_peak_gpu_vram_mb(self) -> float:
        if torch.cuda.is_available():
            return torch.cuda.max_memory_allocated() / (1024 * 1024)
        return 0.0

    def record_memory_snapshot(
        self,
        precision_bits: int,
        checkpoint_dir: str,
        output_csv_path: str = "results/memory.csv"
    ) -> Dict[str, Any]:
        os.makedirs(os.path.dirname(output_csv_path) if os.path.dirname(output_csv_path) else ".", exist_ok=True)
        cpu_ram = self.get_cpu_ram_mb()
        gpu_vram = self.get_gpu_vram_mb()
        peak_vram = self.get_peak_gpu_vram_mb()
        dir_size_bytes = 0
        if os.path.exists(checkpoint_dir):
            for root, _, files in os.walk(checkpoint_dir):
                for f in files:
                    dir_size_bytes += os.path.getsize(os.path.join(root, f))
        checkpoint_size_mb = dir_size_bytes / (1024 * 1024)
        row = {
            "precision_bits": precision_bits,
            "cpu_ram_mb": round(cpu_ram, 2),
            "gpu_vram_mb": round(gpu_vram, 2),
            "peak_gpu_vram_mb": round(peak_vram, 2),
            "checkpoint_size_mb": round(checkpoint_size_mb, 2)
        }
        file_exists = os.path.exists(output_csv_path)
        with open(output_csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)
        return row
'''

# ── upr/layer_hooks.py ───────────────────────────────────────────────────────
files['layer_hooks.py'] = '''\
import torch
import torch.nn as nn
from typing import Dict, List, Any
from .numerical import compute_numerical_metrics

class LayerActivationCollector:
    """Fix 3 — Attaches forward hooks to Transformer blocks to capture activations."""
    def __init__(self, model: nn.Module):
        self.model = model
        self.hooks: List[Any] = []
        self.activations: Dict[str, torch.Tensor] = {}

    def register_hooks(self) -> None:
        self.clear()
        for name, module in self.model.named_modules():
            if any(k in name for k in ["layers.", "block", "h.", "self_attn", "mlp"]):
                hook = module.register_forward_hook(self._make_hook(name))
                self.hooks.append(hook)

    def _make_hook(self, layer_name: str):
        def hook(module, input_tensor, output_tensor):
            inp = input_tensor[0] if isinstance(input_tensor, tuple) and len(input_tensor) > 0 else input_tensor
            out = output_tensor[0] if isinstance(output_tensor, tuple) and len(output_tensor) > 0 else output_tensor
            if isinstance(inp, torch.Tensor):
                self.activations[f"{layer_name}.input"] = inp.detach().cpu()
            if isinstance(out, torch.Tensor):
                self.activations[f"{layer_name}.output"] = out.detach().cpu()
        return hook

    def clear(self) -> None:
        for h in self.hooks:
            h.remove()
        self.hooks = []
        self.activations = {}

def compare_layer_activations(orig_collector: LayerActivationCollector, recon_collector: LayerActivationCollector) -> Dict[str, Any]:
    """Fix 3 — Compares original vs BitPlane layer activations and returns metrics."""
    layer_metrics = {}
    common_keys = set(orig_collector.activations.keys()).intersection(set(recon_collector.activations.keys()))
    for key in sorted(common_keys):
        orig_act = orig_collector.activations[key]
        recon_act = recon_collector.activations[key]
        layer_metrics[key] = compute_numerical_metrics(orig_act, recon_act)
    return layer_metrics
'''

# ── upr/metrics.py ───────────────────────────────────────────────────────────
files['metrics.py'] = '''\
from .numerical import compute_cosine_similarity, compute_kl_divergence, compute_numerical_metrics

def compute_weight_metrics(original, reconstructed):
    """Alias mapping to compute_numerical_metrics for backward compatibility."""
    m = compute_numerical_metrics(original, reconstructed)
    return {
        "exact_match": m["torch_equal"],
        "mae": m["mae"],
        "rmse": m["rmse"],
        "max_error": m["max_abs_error"],
        "mean_relative_error": m["mean_relative_error"],
        "cosine_similarity": m["cosine_similarity"],
        "kl_divergence": m["kl_divergence"],
        "num_elements": m["num_elements"]
    }
'''

# ── Write all files ───────────────────────────────────────────────────────────
written = []
for filename, content in files.items():
    path = os.path.join(UPR_DIR, filename)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    written.append(filename)

print('Bootstrap complete. Written to', UPR_DIR)
print('Files written:')
for w in written:
    print(f'  ✓ upr/{w}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bootstrap complete. Written to /content/drive/MyDrive/UniversalPrecisionRuntime/upr
Files written:
  ✓ upr/__init__.py
  ✓ upr/bit_ops.py
  ✓ upr/numerical.py
  ✓ upr/metadata.py
  ✓ upr/profiler.py
  ✓ upr/layer_hooks.py
  ✓ upr/metrics.py


In [2]:
# Verify the import works
import sys
PROJECT_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import importlib
import upr
importlib.reload(upr)
upr.set_seed(42)
print(f'upr {upr.__version__} imported successfully.')
print('set_seed(42) called — ready to run Notebooks 01 → 04.')

upr 0.1.0 imported successfully.
set_seed(42) called — ready to run Notebooks 01 → 04.
